# Embedder Exploration

This Notebook contains explorations into the usage of the siglip2 embedding model for embedding images into a latent space such that similar objects can be zero-shot classified togethered.

In [1]:
# Install Dependencies
%pip install transformers torch torchvision accelerate matplotlib scikit-learn numpy pandas mediapipe opencv-python chromadb pillow

  Using cached mediapipe-0.10.21-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (9.7 kB)
Using cached mediapipe-0.10.21-cp312-cp312-manylinux_2_28_x86_64.whl (35.6 MB)
Note: you may need to restart the kernel to use updated packages.


## Constants

In [2]:
# MODEL = "google/siglip2-so400m-patch14-384"
# MODEL = "facebook/dinov2-base"
MODEL = "openai/clip-vit-base-patch32"

## Embedding a Single Image

In [ ]:
import torch
from transformers import AutoModel, AutoProcessor
from transformers.image_utils import load_image
import time

model = AutoModel.from_pretrained(MODEL, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL)

start = time.time()
image = load_image("./images/rubiks-cube.jpg")
print(f"Image Load Time: {time.time() - start}")

start = time.time()
inputs = processor(images=[image], return_tensors="pt").to(model.device)
print(f"Preprocessor Time: {time.time() - start}")

start = time.time()
with torch.no_grad():
    image_embeddings = model.get_image_features(**inputs)
print(f"Interence Elapsed Time: {time.time() - start}")

print(image_embeddings.shape)

## Batch Embedding

In [ ]:
import torch
from transformers import AutoModel, AutoProcessor
from transformers.image_utils import load_image
import matplotlib.pyplot as plt
import time

model = AutoModel.from_pretrained(MODEL, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL)

x_axis = [1, 5, 10, 20, 50]
load_times = []
preprocessing_times = []
embedding_times = []

for value in x_axis:
    print(f"Working with {value} images")
    images = []
    start = time.time()
    for _ in range(value):
        images.append(load_image("./images/rubiks-cube.jpg"))
    load_times.append(time.time() - start)

    start = time.time()
    inputs = processor(images=images, return_tensors="pt").to(model.device)
    preprocessing_times.append(time.time() - start)

    start = time.time()
    with torch.no_grad():
        image_embeddings = model.get_image_features(**inputs)
    embedding_times.append(time.time() - start)

plt.subplot(221)
plt.plot(x_axis, load_times)
plt.title("Load Time (s) vs Number of Images")
plt.grid(True)

plt.subplot(222)
plt.plot(x_axis, preprocessing_times)
plt.title("Preprocessing Time (s) vs Number of Images")
plt.grid(True)

plt.subplot(223)
plt.plot(x_axis, embedding_times)
plt.title("Embedding Times (s) vs Number of Images")
plt.grid(True)

print(f"Number of Images: {x_axis}")
print(f"Load Times: {load_times}")
print(f"Preprocessing Times: {preprocessing_times}")
print(f"Embedding Times: {embedding_times}")

# Embedding Pick Items Without Hand Clipping / Masking

In [ ]:
import torch
from transformers import AutoModel, AutoProcessor
from transformers.image_utils import load_image
import os
import numpy as np

model = AutoModel.from_pretrained(MODEL, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL)

images = [load_image(f"../../images/pick-items/{image}") for image in os.listdir("../../images/pick-items")]

inputs = processor(images=images, return_tensors="pt").to(model.device)

with torch.no_grad():
    embeddings = model.get_image_features(**inputs)

embeddings = embeddings.numpy()

shortest_distances = []
for i in range(len(images)):
    shortest_distance = 1e99
    for j in range(len(images)):
        if i == j:
            continue
        dist = np.linalg.norm(embeddings[j] - embeddings[i])
        if dist < shortest_distance:
            shortest_distance = dist
    shortest_distances.append(shortest_distance)

print(f"Shortest Distances: {shortest_distances}")


## Embedding Pick Items with Hand Clipping

In [3]:
import numpy as np

def hand_pos(landmarks, image):
    if len(landmarks) < 21:
        # need at least 21 for a hand
        return ()

    # center of palm appears to be a better indicator for the location of the hand (more mass concentrated at that point)
    fingers = [[np.array(landmarks[i]) for i in range(j * 4 + 1, j * 4 + 5)] for j in range(5)]
    # base of the palm
    total_x = landmarks[0][0] * image.shape[1]
    total_y = landmarks[0][1] * image.shape[0]
    for finger in fingers:
        # look at base of the finger
        curr = finger[0]
        total_x += (curr[0] * image.shape[1])
        total_y += (curr[1] * image.shape[0])
    return (total_x / 6, total_y / 6)

def hand_bounding_box(landmarks, image):
    left = image.shape[1]
    right = -1
    top = image.shape[0]
    bottom = -1
    for (x, y, _) in landmarks:
        #adjust for pixel values
        x = x * image.shape[1]
        y = y * image.shape[0]
        if x < left:
            # point further left than current left
            left = x
        if x > right:
            # point further right than current right
            right = x
        if y < top:
            #point that is higher than top
            top = y
        if y > bottom:
            bottom = y

    #add a little buffer for the rest of the finger (the landmarks aren't exactly at the tip)
    top, bottom, left, right = int(top - 20), int(bottom + 20), int(left - 20), int(right + 20)
    # return points (top left and clockwise from there, (x, y) for points), size (height, width)
    return ((left, top), (right, top), (right, bottom), (left, bottom)), (bottom - top, right - left)


In [4]:
import mediapipe as mp
from matplotlib import pyplot as plt
import cv2
import copy

def draw_hand(image):
    mp_drawing = mp.solutions.drawing_utils
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.3, max_num_hands=2)
    results = hands.process(image)
    draw_image = copy.deepcopy(image)

    hand_points = []
    if results.multi_hand_landmarks:
        for (i, hand_landmarks) in enumerate(results.multi_hand_landmarks):
            mp_drawing.draw_landmarks(draw_image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            points = []
            for i in hand_landmarks.landmark[:21]:
                points.append([i.x, i.y, i.z])
            hand_points.append(points)

    hand_positions = [hand_pos(hand_point, image) for hand_point in hand_points]
    left_hand_points = None
    left_hand_position = (1e99, 1e99)
    for (i, hand_position) in enumerate(hand_positions):
        if hand_position[0] < left_hand_position[0]:
            left_hand_position = hand_position
            left_hand_points = hand_points[i]

    bounding_box, bounding_box_size = hand_bounding_box(left_hand_points, image)
    for point_index in range(len(bounding_box)):
        cv2.line(draw_image, bounding_box[point_index], bounding_box[(point_index + 1) % len(bounding_box)], (0, 0, 255), 4)

    plt.imshow(draw_image)
    plt.show()

    plt.imshow(image[bounding_box[0][1]:bounding_box[2][1], bounding_box[0][0]:bounding_box[2][0]])
    plt.show()

    return image[bounding_box[0][1]:bounding_box[2][1], bounding_box[0][0]:bounding_box[2][0]]


In [ ]:
import cv2
from matplotlib import pyplot as plt

image = cv2.imread("../../images/pick-items/alligatorclip.JPG")
image = cv2.resize(image, (1920, 1080))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.imshow(image)
plt.show()
hand = draw_hand(image)

In [5]:
import mediapipe as mp

def segment_hand(image):
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.3, max_num_hands=2)
    results = hands.process(image)

    hand_points = []
    if results.multi_hand_landmarks:
        for (i, hand_landmarks) in enumerate(results.multi_hand_landmarks):
            points = []
            for i in hand_landmarks.landmark[:21]:
                points.append([i.x, i.y, i.z])
            hand_points.append(points)
    else:
        return None

    if len(hand_points) > 1:
        hand_positions = [hand_pos(hand_point, image) for hand_point in hand_points]
        right_hand_points = None
        right_hand_position = (1e99, 1e99)
        for (i, hand_position) in enumerate(hand_positions):
            if hand_position[0] < right_hand_position[0]:
                right_hand_position = hand_position
                right_hand_points = hand_points[i]
    else:
        right_hand_points = hand_points[0]

    bounding_box, _ = hand_bounding_box(right_hand_points, image)

    return image[bounding_box[0][1]:bounding_box[2][1], bounding_box[0][0]:bounding_box[2][0]]


In [6]:
import numpy as np

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [ ]:
import torch
from transformers import AutoModel, AutoProcessor
from transformers.image_utils import load_image
import os
import numpy as np
from matplotlib import pyplot as plt
import cv2

model = AutoModel.from_pretrained(MODEL, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL)

# images = [load_image(f"../../images/pick_items/{image}") for image in os.listdir("../../images/pick-items")]

# Form a box around the hand
images = []
for f in os.listdir("../../images/pick-items"):
    image = cv2.imread(f"../../images/pick-items/{f}")
    image = cv2.resize(image, (1920, 1080))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = segment_hand(image)
    images.append(image)

# Show the images
for image in images:
    plt.imshow(image)
    plt.show()

inputs = processor(images=images, return_tensors="pt").to(model.device)

with torch.no_grad():
    # outputs = model(**inputs)
    # embeddings = outputs.last_hidden_state.mean(dim=1)
    embeddings = model.get_image_features(**inputs)

# embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
embeddings = embeddings.numpy()

shortest_distances = []
for i in range(len(images)):
    shortest_distance = [1e99, -1]
    for j in range(len(images)):
        if i == j:
            continue
        dist = cosine_similarity(embeddings[j], embeddings[i])
        if dist < shortest_distance[0]:
            shortest_distance = [dist, i]
    shortest_distances.append(shortest_distance)

print(f"Shortest Distances: {shortest_distances}")

farthest_distances = []
for i in range(len(images)):
    farthest_distance = [-1e99, -1]
    for j in range(len(images)):
        if i == j:
            continue
        dist = cosine_similarity(embeddings[j], embeddings[i])
        if dist > farthest_distance[0]:
            farthest_distance = [dist, i]
    farthest_distances.append(farthest_distance)

print(f"Farthest Distances: {farthest_distances}")

# Different Backgrounds

In [ ]:
import torch
from transformers import AutoModel, AutoProcessor
import os
import numpy as np
from matplotlib import pyplot as plt
import cv2

model = AutoModel.from_pretrained(MODEL, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL)

images = []
for f in os.listdir("../../images/background-test"):
    image = cv2.imread(f"../../images/background-test/{f}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = segment_hand(image)
    image = cv2.resize(image, (1200, 1200))
    images.append(image)

# Show the images
for image in images:
    plt.imshow(image)
    plt.show()

inputs = processor(images=images, return_tensors="pt").to(model.device)

with torch.no_grad():
    # outputs = model(**inputs)
    # embeddings = outputs.last_hidden_state.mean(dim=1)
    embeddings = model.get_image_features(**inputs)

# embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
embeddings = embeddings.numpy()

shortest_distances = []
for i in range(len(images)):
    shortest_distance = [1e99, -1]
    for j in range(len(images)):
        if i == j:
            continue
        dist = cosine_similarity(embeddings[j], embeddings[i])
        if dist < shortest_distance[0]:
            shortest_distance = [dist, i]
    shortest_distances.append(shortest_distance)

print(f"Shortest Distances: {shortest_distances}")

farthest_distances = []
for i in range(len(images)):
    farthest_distance = [-1e99, -1]
    for j in range(len(images)):
        if i == j:
            continue
        dist = cosine_similarity(embeddings[j], embeddings[i])
        if dist > farthest_distance[0]:
            farthest_distance = [dist, i]
    farthest_distances.append(farthest_distance)

print(f"Farthest Distances: {farthest_distances}")

In [ ]:
import torch
from transformers import AutoModel, AutoProcessor
import os
import numpy as np
from matplotlib import pyplot as plt
import cv2

model = AutoModel.from_pretrained(MODEL, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL)

images = []
for f in os.listdir("../../images/orientation-test"):
    image = cv2.imread(f"../../images/orientation-test/{f}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = segment_hand(image)
    image = cv2.resize(image, (1200, 1200))
    images.append(image)

# Show the images
for image in images:
    plt.imshow(image)
    plt.show()

inputs = processor(images=images, return_tensors="pt").to(model.device)

with torch.no_grad():
    # outputs = model(**inputs)
    # embeddings = outputs.last_hidden_state.mean(dim=1)
    embeddings = model.get_image_features(**inputs)

# embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
embeddings = embeddings.numpy()

shortest_distances = []
for i in range(len(images)):
    shortest_distance = [1e99, -1]
    for j in range(len(images)):
        if i == j:
            continue
        dist = cosine_similarity(embeddings[j], embeddings[i])
        if dist < shortest_distance[0]:
            shortest_distance = [dist, i]
    shortest_distances.append(shortest_distance)

print(f"Shortest Distances: {shortest_distances}")

farthest_distances = []
for i in range(len(images)):
    farthest_distance = [-1e99, -1]
    for j in range(len(images)):
        if i == j:
            continue
        dist = cosine_similarity(embeddings[j], embeddings[i])
        if dist > farthest_distance[0]:
            farthest_distance = [dist, i]
    farthest_distances.append(farthest_distance)

print(f"Farthest Distances: {farthest_distances}")

# Embedding Picklists

In [ ]:
import torch
from transformers import AutoModel, AutoProcessor
import os
import numpy as np
import cv2
import chromadb

model = AutoModel.from_pretrained(MODEL, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL)
chroma_client = chromadb.Client()

chroma_client.delete_collection(name="embedding_picklist_test")
collection = chroma_client.create_collection(
    name="embedding_picklist_test",
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    },
    metadata={
        "description": "test database for testing my model's ability to map images to names"
    }
)
id = 1

In [ ]:
def embed(image, picklist, id: int, model, processor, collection):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = segment_hand(image)
    if image is None:
        print("Unable to Segment Image")
        return
    inputs = processor(images=[image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        embeddings = model.get_image_features(**inputs)
    collection.add(
        ids=[f"id{id}"],
        embeddings=embeddings.numpy(),
        metadatas=[{"picklist": picklist}]
    )
    return embeddings.numpy()

In [ ]:
picklists = ["gbr", "b", "br"]

# Embed picklists
for (i, picklist) in enumerate(picklists):
    for f in os.listdir(f"../../images/picklist-test/{i}"):
        image = cv2.imread(f"../../images/picklist-test/{i}/{f}")
        embed(image, picklist, id)
        id += 1


In [ ]:
image = cv2.imread("../../images/picklist-test/test/b.png")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image = segment_hand(image)
query = None
if image is not None:
    inputs = processor(images=[image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        embeddings = model.get_image_features(**inputs)
    query = collection.query(
        query_embeddings=embeddings.numpy(),
    )
else:
    print("Could not find hands")
print(query)

In [ ]:
image = cv2.imread("../../images/picklist-test/test/r.png")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image = segment_hand(image)
query = None
if image is not None:
    inputs = processor(images=[image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        embeddings = model.get_image_features(**inputs)
    query = collection.query(
        query_embeddings=embeddings.numpy(),
    )
else:
    print("Could not find hands")
print(query)

In [ ]:
image = cv2.imread("../../images/picklist-test/test/g.png")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image = segment_hand(image)
query = None
if image is not None:
    inputs = processor(images=[image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        embeddings = model.get_image_features(**inputs)
    query = collection.query(
        query_embeddings=embeddings.numpy(),
    )
else:
    print("Could not find hands")
print(query)

# Creating heatmaps for our objects

In [ ]:
from transformers import AutoModel, AutoProcessor
import chromadb

model = AutoModel.from_pretrained(MODEL, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL)
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(name="all_embeddings")
except:
    pass
collection = chroma_client.create_collection(
    name="all_embeddings",
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    },
    metadata={
        "description": "test dataset for embedding all of the items we have"
    }
)
id = 1

In [ ]:
import torch
import os
import cv2

def embed_and_process_img(image_name, id: int, model, processor, collection):
    image = cv2.imread(image_name)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = segment_hand(image)
    if image is None:
        print(f"Unable to segment image in {image_name}")
        return (None, None)
    inputs = processor(images=[image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        embeddings = model.get_image_features(**inputs)
    collection.add(
        ids=[f"id{id}"],
        embeddings=embeddings.numpy()
    )
    return (embeddings.numpy(), cv2.resize(image, (64, 64)))


embeddings = []
images = []
image_names = []
for f in os.listdir("../../images/our-items/"):
    print(f"Embedding: {f}")
    (embedding, image) = embed_and_process_img(f"../../images/our-items/{f}", id, model, processor, collection)
    if embedding is not None and image is not None:
        embeddings.append(embedding)
        images.append(image)
        image_names.append(f.split(".")[0])
    id += 1

In [ ]:
print(f"Embeddings: {len(embeddings)}, Images: {len(images)}, Image Names: {len(image_names)}")
images = np.array(images)

In [ ]:
cosine_similarities = np.zeros((len(embeddings), len(embeddings)))

for i in range(len(embeddings)):
    for j in range(len(embeddings)):
        cosine_similarities[i, j] = cosine_similarity(embeddings[i].reshape(512), embeddings[j].reshape(512))

np.min(cosine_similarities)

In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

fig, ax = plt.subplots(figsize=(20, 20))
im = ax.imshow(cosine_similarities[:10, :10], vmin=0.6, vmax=1.0)

ax.set_xticks(range(10), labels=image_names[:10], rotation=45, ha="right", rotation_mode="anchor")
ax.set_yticks(range(10), labels=image_names[:10], rotation=45)

for y in range(10):
    imagebox = OffsetImage(images[y])
    ab = AnnotationBbox(
        imagebox,
        xy=(-0.1, -y / 10 + 0.95),
        xycoords='axes fraction',
        frameon=False
    )
    ax.add_artist(ab)

for x in range(10):
    imagebox = OffsetImage(images[x])
    ab = AnnotationBbox(
        imagebox,
        xy=(x / 10 + 0.05, -0.1),
        xycoords='axes fraction',
        frameon=False
    )
    ax.add_artist(ab)

for i in range(10):
    for j in range(10):
        text = ax.text(i, j, f"{round(cosine_similarities[i, j], 3)}", ha="center", va="center", color="w")

ax.set_title("Cosine Similarities")
fig.tight_layout()
plt.show()

In [ ]:
for x in range(0, len(image_names), 10):
    for y in range(0, len(image_names), 10):
        fig, ax = plt.subplots(figsize=(20, 20))
        im = ax.imshow(cosine_similarities[y:(y+10), x:(x+10)], vmin=0.6, vmax=1.0)

        ax.set_xticks(range(len(image_names[x:(x+10)])), labels=image_names[x:(x+10)], rotation=45, ha="right", rotation_mode="anchor")
        ax.set_yticks(range(len(image_names[y:(y+10)])), labels=image_names[y:(y+10)], rotation=45)

        for i in range(len(image_names[y:(y+10)])):
            imagebox = OffsetImage(images[y + i])
            ab = AnnotationBbox(
                imagebox,
                xy=(-1 / len(image_names[x:(x+10)]), -i / len(image_names[y:(y+10)]) + 1 - 1 / (2 * len(image_names[y:(y+10)]))),
                xycoords='axes fraction',
                frameon=False
            )
            ax.add_artist(ab)

        for j in range(len(image_names[x:(x+10)])):
            imagebox = OffsetImage(images[x + j])
            ab = AnnotationBbox(
                imagebox,
                xy=(j / len(image_names[x:(x+10)]) + 1 / (2 * len(image_names[x:(x+10)])), -1 / len(image_names[y:(y+10)])),
                xycoords='axes fraction',
                frameon=False
            )
            ax.add_artist(ab)

        for i in range(len(image_names[x:(x+10)])):
            for j in range(len(image_names[y:(y+10)])):
                print(f"({x + i}, {y + j}): {cosine_similarities[x + i, y + j]}")
                text = ax.text(i, j, f"{round(cosine_similarities[x + i, y + j], 3)}", ha="center", va="center", color="w")
        
        ax.set_title(f"Cosine Similarities (x=[{x}, {x+10}]) (y=[{y}, {y+10}])")
        fig.tight_layout()
        plt.savefig(f"./heatmaps/cosine-similarities-{x}-{y}.png")
        plt.clf()
        plt.close()


In [ ]:
closest_images = []
for i in range(len(image_names)):
    closest_image = np.argmax(cosine_similarities[i, [False if idx == i else True for idx in range(len(image_names))]])
    closest_images.append((image_names[i], image_names[closest_image], cosine_similarities[i, closest_image]))

In [ ]:
closest_images

In [ ]:
print(collection.count())
results = collection.query(query_embeddings=embeddings[0], n_results=10)
results

# TESTING THE DETECTION ALGORITHM

In [7]:
from transformers import AutoModel, AutoProcessor
import chromadb

model = AutoModel.from_pretrained(MODEL, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL)
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(name="detection_testing")
except:
    pass
collection = chroma_client.create_collection(
    name="detection_testing",
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    },
    metadata={
        "description": "test dataset for embedding images so we can test our detection algorithm"
    }
)
id = 1

DETECT_THRESHOLD = 0.05

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [8]:
import torch
import cv2

def embed_image(image_name: str, id: int, picklist: str, model, processor, collection):
    image = cv2.imread(image_name)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = segment_hand(image)
    if image is None:
        print(f"Unable to segment image in {image_name}")
        return None
    inputs = processor(images=[image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        embeddings = model.get_image_features(**inputs)
    collection.add(
        ids=[f"id{id}"],
        embeddings=embeddings.numpy(),
        metadatas=[{"picklist": picklist}]
    )
    return embeddings.numpy()

In [9]:
def perform_inference(embedding, collection):
    results = collection.query(query_embeddings=embedding)
    print(results)
    picklists = [results["metadatas"][0][i]["picklist"] for i in range(len(results["ids"][0])) if results["distances"][0][i] <= DETECT_THRESHOLD]
    bins = {}
    for picklist in picklists:
        for c in picklist:
            if c in bins:
                bins[c] += 1
            else:
                bins[c] = 1
    return (max(bins, key=bins.get), len(picklists))


In [10]:
import os
os.environ["GLOG_minloglevel"] = "3"   # 0=INFO, 1=WARNING, 2=ERROR
os.environ["absl_log_level"] = "3"

picklists = ["gbr", "b", "br"]

for (i, picklist) in enumerate(picklists):
    for f in os.listdir(f"../../images/picklist-test/{i}"):
        embedding = embed_image(
            f"../../images/picklist-test/{i}/{f}",
            id,
            picklist,
            model,
            processor,
            collection
        )
        id += 1
        (inference, total_results) = perform_inference(embedding, collection)
        print(f"Picklist: {picklist}, File: {f}, Inference: {inference}, Total Results: {total_results}")

I0000 00:00:1763504784.068869 1502633 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1763504784.073532 1502998 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: GFX1103_R1 (gfx1103_r1, LLVM 15.0.7, DRM 3.57, 6.8.0-87-generic)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1763504784.110981 1502976 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1763504784.138042 1502982 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1763504784.152393 1502986 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


{'ids': [['id1']], 'embeddings': None, 'documents': [[None]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'picklist': 'gbr'}]], 'distances': [[-4.76837158203125e-07]]}
Picklist: gbr, File: 0.png, Inference: g, Total Results: 1


I0000 00:00:1763504784.346160 1502633 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1763504784.347651 1503052 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: GFX1103_R1 (gfx1103_r1, LLVM 15.0.7, DRM 3.57, 6.8.0-87-generic)
W0000 00:00:1763504784.386692 1503037 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1763504784.410836 1503035 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


{'ids': [['id2', 'id1']], 'embeddings': None, 'documents': [[None, None]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'picklist': 'gbr'}, {'picklist': 'gbr'}]], 'distances': [[-3.5762786865234375e-07, 0.16526752710342407]]}
Picklist: gbr, File: 2.png, Inference: g, Total Results: 1


I0000 00:00:1763504784.592501 1502633 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1763504784.593827 1503071 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: GFX1103_R1 (gfx1103_r1, LLVM 15.0.7, DRM 3.57, 6.8.0-87-generic)
W0000 00:00:1763504784.630810 1503056 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1763504784.662197 1503059 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


...

I0000 00:00:1763504784.846786 1502633 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1763504784.848667 1503090 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: GFX1103_R1 (gfx1103_r1, LLVM 15.0.7, DRM 3.57, 6.8.0-87-generic)
W0000 00:00:1763504784.889292 1503078 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1763504784.914631 1503077 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


{'ids': [['id4', 'id1', 'id3', 'id2']], 'embeddings': None, 'documents': [[None, None, None, None]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'picklist': 'b'}, {'picklist': 'gbr'}, {'picklist': 'gbr'}, {'picklist': 'gbr'}]], 'distances': [[-7.152557373046875e-07, 0.04651874303817749, 0.17028146982192993, 0.21812736988067627]]}
Picklist: b, File: 0.png, Inference: b, Total Results: 2


I0000 00:00:1763504785.097448 1502633 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1763504785.100015 1503109 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: GFX1103_R1 (gfx1103_r1, LLVM 15.0.7, DRM 3.57, 6.8.0-87-generic)
W0000 00:00:1763504785.139621 1503095 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1763504785.164705 1503104 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


{'ids': [['id5', 'id1', 'id4', 'id3', 'id2']], 'embeddings': None, 'documents': [[None, None, None, None, None]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'picklist': 'br'}, {'picklist': 'gbr'}, {'picklist': 'b'}, {'picklist': 'gbr'}, {'picklist': 'gbr'}]], 'distances': [[4.76837158203125e-07, 0.046482205390930176, 0.08829909563064575, 0.17030715942382812, 0.18149489164352417]]}
Picklist: br, File: 0.png, Inference: b, Total Results: 2
{'ids': [['id6', 'id2', 'id5', 'id1', 'id3', 'id4']], 'embeddings': None, 'documents': [[None, None, None, None, None, None]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'picklist': 'br'}, {'picklist': 'gbr'}, {'picklist': 'br'}, {'picklist': 'gbr'}, {'picklist': 'gbr'}, {'picklist': 'b'}]], 'distances': [[-5.960464477539062e-07, 0.08760654926300049, 0.14560139179229736, 0.15667486190795898, 0.1660662293434143, 0.20765036344528198]]}
Picklist: br

I0000 00:00:1763504785.350719 1502633 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1763504785.352104 1503128 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: GFX1103_R1 (gfx1103_r1, LLVM 15.0.7, DRM 3.57, 6.8.0-87-generic)
W0000 00:00:1763504785.391984 1503112 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1763504785.422356 1503115 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
